In [ ]:
!pip install -q -U keras keras-hub keras-nlp datasets jax jaxlib
!pip install -q -U "jax[cuda12]"

In [ ]:
import os
import gc
from kaggle_secrets import UserSecretsClient

# 1. AUTENTICAÇÃO SEGURA NATIVA DO KAGGLE
user_secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

# Obriga o JAX a pedir memória apenas quando precisar, em vez de sequestrar tudo
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

os.environ["KERAS_BACKEND"] = "jax"






In [ ]:
import keras
import keras_hub

# Mantemos bfloat16 para caber nos 16GB da GPU T4
keras.config.set_floatx("bfloat16")
print(f"✅ Backend: {keras.config.backend()} | Precisão: {keras.config.floatx()}")

# Habilita o paralelismo de dados nas 2 GPUs T4 do Kaggle automaticamente. 
# Usando 2 GPUs o batch_size precisa ser múltiplo de 2

#keras.distribution.set_distribution(keras.distribution.DataParallel())




In [ ]:
# 2. CARREGANDO O MODELO BASE
print("\n⏳ Carregando o Gemma 2B Base...")
gemma_lm = keras_hub.models.GemmaCausalLM.from_preset("gemma_2b_en")

In [ ]:
from datasets import load_dataset


print("\n📥 Baixando e processando o Medical Meadows...")
dataset_hf = load_dataset("medalpaca/medical_meadow_medical_flashcards", split="train")
amostra_dataset = dataset_hf.select(range(500))

template = "Instruction:\n{instruction}\n\nResponse:\n{output}"
dataset_treino = [
    template.format(instruction=item["instruction"], output=item["output"])
    for item in amostra_dataset
]

del amostra_dataset
gc.collect()

In [ ]:
#3. Prompt zeroshot sem otimização
prompt_clinico = (
    "Instruction:\nPaciente de 65 anos, histórico de ICFER, "
    "apresenta dispneia paroxística noturna, estertores crepitantes bibasais "
    "e BNP de 850 pg/mL. PA 90/60 mmHg. Qual a conduta farmacológica inicial?\n\n"
    "Response:\n"
)

# Testando o modelo antes do fine-tuning (Zero-Shot)
print("Resposta do Gemma Genérico:")
print(gemma_lm.generate(prompt_clinico, max_length=256))

In [ ]:
# 3. ADAPTAÇÃO DE DOMÍNIO (LORA)
print("\n📉 Habilitando LoRA...")
gemma_lm.backbone.enable_lora(rank=8)

print("\n🚀 Compilando e Treinando (JAX XLA ativado)...")
gemma_lm.preprocessor.sequence_length = 128 

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.AdamW(
        learning_rate=2e-5,
        weight_decay=0.01,
        clipnorm = 1.0
    ),
    weighted_metrics=["accuracy"],
)

gemma_lm.fit(dataset_treino, epochs=2, batch_size=4)


In [ ]:
#3. Prompt zeroshot após três épocas
prompt_clinico = (
    "Instruction:\nPaciente de 65 anos, histórico de ICFER, "
    "apresenta dispneia paroxística noturna, estertores crepitantes bibasais "
    "e BNP de 850 pg/mL. PA 90/60 mmHg. Qual a conduta farmacológica inicial?\n\n"
    "Response:\n"
)

# Testando o modelo pós fine-tuning (Zero-Shot)
print("Resposta do Gemma com fine-tuning:")
print(gemma_lm.generate(prompt_clinico, max_length=256))

In [ ]:
# 4. INFERÊNCIA E EXPORTAÇÃO
print("\n✅ Teste Pós-Treinamento:")
prompt_clinico = (
    "Instruction:\nPaciente de 65 anos, histórico de ICFER, "
    "apresenta dispneia paroxística noturna, estertores crepitantes bibasais "
    "e BNP de 850 pg/mL. PA 90/60 mmHg. Qual a conduta farmacológica inicial?\n\n"
    "Response:\n"
)
print(gemma_lm.generate(prompt_clinico, max_length=256))

gemma_lm.save("gemma_medico_lora.keras")

